In [19]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

import plotly.express as px

In [3]:
df.shape
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Transaction_ID        15000 non-null  object 
 1   Timestamp             15000 non-null  object 
 2   Customer_ID           15000 non-null  object 
 3   Amount                15000 non-null  float64
 4   Merchant_Category     15000 non-null  object 
 5   Distance_from_Home    15000 non-null  float64
 6   Device_Type           15000 non-null  object 
 7   IP_Risk_Score         15000 non-null  float64
 8   Avg_Spending_Habit    15000 non-null  float64
 9   Is_Weekend            15000 non-null  int64  
 10  Is_Night_Transaction  15000 non-null  int64  
 11  Is_Fraud              15000 non-null  int64  
dtypes: float64(4), int64(3), object(5)
memory usage: 1.4+ MB


Transaction_ID          0
Timestamp               0
Customer_ID             0
Amount                  0
Merchant_Category       0
Distance_from_Home      0
Device_Type             0
IP_Risk_Score           0
Avg_Spending_Habit      0
Is_Weekend              0
Is_Night_Transaction    0
Is_Fraud                0
dtype: int64

In [4]:
features = [
    "Amount",
    "Distance_from_Home",
    "IP_Risk_Score",
    "Avg_Spending_Habit",
    "Is_Weekend",
    "Is_Night_Transaction"
]

X = df[features]

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [6]:
model = IsolationForest(
    contamination=0.03,
    random_state=42
)

model.fit(X_scaled)

IsolationForest(contamination=0.03, random_state=42)

In [7]:
df["anomaly_score"] = model.decision_function(X_scaled)
df["prediction"] = model.predict(X_scaled)

df["risk"] = df["prediction"].map({1: "Normal", -1: "Suspicious"})
df["risk"].value_counts()

risk
Normal        14550
Suspicious      450
Name: count, dtype: int64

In [8]:
def build_evidence(row):
    evidence = []

    if row["Amount"] > row["Avg_Spending_Habit"] * 3:
        evidence.append("Transaction amount is significantly higher than the customer's normal spending pattern.")

    if row["Distance_from_Home"] > 50:
        evidence.append("Transaction occurred far from customer's usual location.")

    if row["IP_Risk_Score"] > 80:
        evidence.append("High-risk IP address detected.")

    if row["Is_Night_Transaction"] == 1:
        evidence.append("Transaction occurred during unusual overnight hours.")

    if row["Is_Weekend"] == 1:
        evidence.append("Transaction occurred on a weekend.")

    if len(evidence) == 0:
        evidence.append("No strong anomalies detected, but transaction deviates slightly from baseline behavior.")

    return evidence

In [9]:
df["evidence"] = df.apply(build_evidence, axis=1)
df[["Amount", "risk", "evidence"]].head()

,Amount,risk,evidence
0,1985.97,Normal,[Transaction amount is significantly higher th...
1,30.45,Normal,"[No strong anomalies detected, but transaction..."
2,10.08,Normal,"[No strong anomalies detected, but transaction..."
3,8.20,Normal,"[No strong anomalies detected, but transaction..."
4,181.07,Normal,[Transaction occurred on a weekend.]


In [10]:
from google import genai
import os
from dotenv import load_dotenv

load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [11]:
sample = df[df["risk"] == "Suspicious"].iloc[0]

sample_evidence = sample["evidence"]
sample_evidence

["Transaction amount is significantly higher than the customer's normal spending pattern.",
 'Transaction occurred during unusual overnight hours.']

In [12]:
prompt = f"""
You are a fraud analyst at a global bank.

You are reviewing a flagged transaction.

Your job is NOT to decide fraud.
Your job is to explain why it needs investigation.

Evidence:
{sample_evidence}

Write a short analyst-style summary (max 120 words).
Include:
- Why it was flagged
- What looks unusual
- Recommended next step
"""

In [13]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

**Analyst Summary: Flagged Transaction Review**

This transaction was flagged due to a clear deviation from the customer's established behavioral profile. The amount is significantly higher than their normal spending patterns, a primary trigger for unusual activity. This is further compounded by the transaction occurring during unusual overnight hours, a time when the customer typically does not transact. The confluence of these two high-risk indicators strongly suggests a need for investigation, potentially indicating account compromise or unauthorized activity.

**Recommended next step:** Initiate immediate customer outreach via verified channels for transaction validation, and consider a temporary hold on funds pending confirmation.
